# LEO Rendezvous and Docking

This notebook demonstrates spacecraft rendezvous operations in Low Earth Orbit, including:
1. Multi-spacecraft simulation setup
2. Phasing maneuvers
3. Relative navigation
4. Docking sequence

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../space_state')

from base import (
    Simulation, SpacecraftEntity, Spacecraft, Thruster,
    get_circular_state, BODIES, get_body_position, get_body_velocity,
)
from base.star_constants import IDX_R, IDX_V, IDX_M, AU_TO_KM

## 1. Define Target and Chaser Spacecraft

- **Target**: Space station (ISS-like)
- **Chaser**: Crew vehicle (Dragon-like)

In [ ]:
# Target spacecraft (station)
target = Spacecraft(
    name="Space Station",
    dry_mass=420000,  # kg (ISS mass)
    propellant_mass=1000,
    thrusters=[],  # Station doesn't maneuver
)

# Chaser spacecraft
chaser_engine = Thruster(
    name="Draco Thrusters",
    thrust_n=16000,  # 16 kN total
    isp_s=300,
)

chaser = Spacecraft(
    name="Crew Vehicle",
    dry_mass=9500,  # kg
    propellant_mass=2500,  # kg
    thrusters=[chaser_engine],
)

print(f"Target: {target._name} ({target.mass:,.0f} kg)")
print(f"Chaser: {chaser._name} ({chaser.mass:,.0f} kg)")
print(f"Chaser delta-V: {chaser.delta_v_remaining:.2f} km/s")

## 2. Initialize Simulation with Both Spacecraft

In [ ]:
# Create simulation
sim = Simulation(name="LEO Rendezvous")

# ISS orbit parameters
altitude_km = 420  # km
inclination = 51.6  # degrees (ISS inclination)
t0 = 0.0

# Target state (at some position in orbit)
target_state = get_circular_state(
    body='earth',
    altitude_km=altitude_km,
    inclination_deg=inclination,
    time=t0,
    mass=target.mass,
)

# Chaser state (same orbit, but 100 km behind)
# In a circular orbit, we can phase the chaser by adjusting its position
# Phase angle for 100 km behind at 420 km altitude
orbital_radius_km = 6371 + altitude_km  # Earth radius + altitude
phase_angle_deg = 100 / (2 * np.pi * orbital_radius_km) * 360  # ~0.85 degrees

chaser_state = get_circular_state(
    body='earth',
    altitude_km=altitude_km,
    inclination_deg=inclination,
    time=t0,
    mass=chaser.mass,
    raan_deg=-phase_angle_deg,  # Behind target
)

# Add both to simulation
sim.add_spacecraft(target, target_state, id="ISS")
sim.add_spacecraft(chaser, chaser_state, id="Dragon")

print(sim.get_summary())

## 3. Check Initial Relative State

In [ ]:
# Get relative state
rel_state = sim.get_relative_state("ISS", "Dragon")

print("Initial Relative State:")
print("=" * 40)
print(f"Range: {rel_state['range_km']:.2f} km")
print(f"Closing rate: {rel_state['closing_rate_mps']:.2f} m/s")
print(f"Relative position (AU): {rel_state['r_rel_au']}")

## 4. Propagate and Track Relative Motion

In [ ]:
# Propagate both spacecraft for several orbital periods
orbital_period_days = 92.68 / (24 * 60)  # ~92.68 minutes in days
t_end = 3 * orbital_period_days  # 3 orbits

# Propagate in small steps to track relative motion
dt = orbital_period_days / 50  # 50 points per orbit
n_steps = int(t_end / dt)

# Store relative state history
relative_history = []
times = [sim.current_time]
relative_history.append(sim.get_relative_state("ISS", "Dragon"))

for i in range(n_steps):
    sim.propagate_steps(dt, n_steps=1)
    times.append(sim.current_time)
    relative_history.append(sim.get_relative_state("ISS", "Dragon"))

# Extract range data
ranges_km = [r['range_km'] for r in relative_history]
closing_rates = [r['closing_rate_mps'] for r in relative_history]

print(f"Propagated for {n_steps} steps ({times[-1]*24*60:.1f} minutes)")
print(f"Final range: {ranges_km[-1]:.2f} km")

## 5. Plot Relative Motion

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Convert times to minutes
times_min = np.array(times) * 24 * 60

# Range vs time
ax = axes[0, 0]
ax.plot(times_min, ranges_km, 'b-', linewidth=2)
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Range (km)')
ax.set_title('Range to Target')
ax.grid(True, alpha=0.3)

# Closing rate vs time
ax = axes[0, 1]
ax.plot(times_min, closing_rates, 'r-', linewidth=2)
ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Closing Rate (m/s)')
ax.set_title('Closing Rate')
ax.axhline(y=0, color='gray', linestyle='--')
ax.grid(True, alpha=0.3)

# Relative trajectory (LVLH frame approximation)
ax = axes[1, 0]
# Extract relative position components in AU
r_rel_x = [r['r_rel_au'][0] for r in relative_history]
r_rel_y = [r['r_rel_au'][1] for r in relative_history]
# Convert to km
r_rel_x_km = np.array(r_rel_x) * AU_TO_KM
r_rel_y_km = np.array(r_rel_y) * AU_TO_KM
ax.plot(r_rel_x_km, r_rel_y_km, 'g-', linewidth=1)
ax.scatter([r_rel_x_km[0]], [r_rel_y_km[0]], c='green', s=100, label='Start', zorder=5)
ax.scatter([r_rel_x_km[-1]], [r_rel_y_km[-1]], c='red', s=100, label='End', zorder=5)
ax.scatter([0], [0], c='blue', s=200, marker='s', label='Target', zorder=5)
ax.set_xlabel('X (km)')
ax.set_ylabel('Y (km)')
ax.set_title('Relative Trajectory (inertial frame)')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Orbit visualization
ax = axes[1, 1]
# Get trajectories from results
iss_entity = sim.get_entity("ISS")
dragon_entity = sim.get_entity("Dragon")

# Get Earth position
r_earth = get_body_position('earth', 0.0)

# Plot relative to Earth
iss_pos = iss_entity.propagator.result.y[:3, :].T - r_earth
dragon_pos = dragon_entity.propagator.result.y[:3, :].T - r_earth

# Convert to km and scale
scale = AU_TO_KM / 1000  # to thousands of km
ax.plot(iss_pos[:, 0] * scale, iss_pos[:, 1] * scale, 'b-', linewidth=2, label='ISS')
ax.plot(dragon_pos[:, 0] * scale, dragon_pos[:, 1] * scale, 'r-', linewidth=1, label='Dragon')
ax.scatter([0], [0], c='lightblue', s=500, label='Earth')
ax.set_xlabel('X (1000 km)')
ax.set_ylabel('Y (1000 km)')
ax.set_title('Orbital Trajectories')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Docking Simulation

When the chaser is close enough, initiate docking.

In [ ]:
# Check current state
rel_state = sim.get_relative_state("ISS", "Dragon")
print(f"Current range: {rel_state['range_km']:.2f} km")
print(f"Closing rate: {rel_state['closing_rate_mps']:.2f} m/s")

# For demonstration, let's simulate docking
# In reality, the chaser would need to match velocity precisely
if rel_state['range_km'] < 200:  # Within 200 km
    print("\nInitiating docking sequence...")
    
    # Perform dock
    combined_id = sim.dock("ISS", "Dragon", new_id="ISS+Dragon")
    
    if combined_id:
        print(f"\nDocking successful!")
        print(sim.get_summary())
else:
    print("\nToo far for docking. Additional phasing required.")

## 7. Relative Navigation Concepts

Understanding the Hill/Clohessy-Wiltshire equations for proximity operations.

In [ ]:
# Hill's equations describe relative motion in the LVLH frame
# For circular orbits, the mean motion is:
mu_earth = BODIES['earth']['mu_km3_s2']
r_orbit = 6371 + altitude_km  # km

n = np.sqrt(mu_earth / r_orbit**3)  # mean motion (rad/s)
T = 2 * np.pi / n  # orbital period (seconds)

print("Orbital Parameters:")
print("=" * 40)
print(f"Mean motion (n): {n:.6f} rad/s")
print(f"Orbital period: {T/60:.2f} minutes")
print(f"")

# Key relationships for rendezvous:
print("Key Rendezvous Relationships:")
print("=" * 40)
print("1. Radial impulse (up/down): Creates elliptical relative motion")
print("2. Along-track impulse (forward/back): Changes altitude, drifts away")
print("3. Cross-track impulse (left/right): Only affects inclination")
print("")
print("V-bar approach: Along velocity vector (safe, standard)")
print("R-bar approach: Along radial vector (direct, more delta-V)")

## Summary

Key concepts demonstrated:
1. Multi-spacecraft simulation with the `Simulation` class
2. Relative state tracking (range, closing rate)
3. Docking operations (combining spacecraft)
4. Orbital mechanics fundamentals for rendezvous